In [ ]:
import torch
torch.set_float32_matmul_precision('high')  # or 'medium' for even faster, lower precision

In [ ]:
import numpy as np
import random
import os

def set_seed(seed: int = 42):
    # Python built-in random module
    random.seed(seed)
    
    # Numpy library
    np.random.seed(seed)
    
    # PyTorch seed for CPU and all GPU devices
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # CuDNN determinism (crucial for GPU reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set a fixed value for the Python hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # print(f"Random seed set as {seed}")

set_seed(42)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3.5-4B",
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-4B",
    dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="eager"
)

In [ ]:
messages = [
    {"role": "user", "content": "When did Virgin Australia start operating?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


In [ ]:
tokenizer.vocab_size

In [ ]:
inputs['input_ids']

In [ ]:
model.eval()

In [ ]:
# from collections import defaultdict
# lookup = defaultdict(dict)

# with torch.no_grad():
#     for tid in range(0, tokens.shape[1]):
#         output = model(input_ids=tokens[:, tid].unsqueeze(0).to(model.device), output_hidden_states=True)
#         for l, hs in enumerate(output.hidden_states):
#             lookup[l][tokens[0, tid].item()] = output.hidden_states[l]

In [ ]:
from collections import defaultdict
lookup = defaultdict(dict)

with torch.no_grad():
    for tid in range(0, tokenizer.vocab_size):
        if tid % 1000 == 0:
            print(tid)
        output = model(input_ids=torch.tensor([1, tid]).unsqueeze(1).to(model.device), output_hidden_states=True)
        for l, hs in enumerate(output.hidden_states):
            lookup[l][tid] = output.hidden_states[l][:, -1, :]

torch.save(dict(lookup), 'llama_lookup.pt')

In [ ]:
lookup = torch.load('llama_lookup.pt')

In [ ]:
# for l in range(len(lookup)):
#     lookup[l] = torch.stack(list(lookup[l].values()), dim=0).squeeze()
# torch.save(lookup, 'llama_lookup.pt')

In [ ]:
with torch.no_grad():
    set_seed(42)
    output = model(**inputs, output_hidden_states=True, output_attentions=True)

In [ ]:
len(output.attentions)

In [ ]:
for l in range(1, 33):
    print(f"Layer {l}")
    for i in range(inputs['input_ids'][0].shape[0]):
        print(f"Token {inputs['input_ids'][0][i].item()}")
        
        x = output.hidden_states[l][:, i, :]
        _, indices = torch.topk(torch.cdist(x, lookup[l]), k=10, largest=False)
        print(indices)
    print(f"------------------------------")
    

In [ ]:
torch.cdist(x, lookup).min()

In [ ]:
torch.cdist(x, lookup.to(torch.float32))[0, 1889]

In [ ]:
output.hidden_states[2][0, 0, :].unsqueeze(0).cpu()

In [ ]:
tokenizer.vocab_size

In [ ]:
token_ids = list(range(tokenizer.vocab_size))

In [ ]:
from common import batch_iter

In [ ]:
for ids in batch_iter(token_ids, 256):
    print(ids)
    print(torch.tensor(ids, dtype=torch.long, device=args.device).unsqueeze(1).shape)
    break